# 02 EEA Batch Ingestion

**Phase:** 3 — EEA Batch Ingestion  
**Status:** Issues 3.1-3.7 complete. Issue 3.8 pending.

This notebook is the readable Phase 3 documentation trail for EEA historical
air quality batch ingestion. It documents source access, raw-data policy,
schema contracts, data quality rules, and output conventions phase by phase
as later issues are resolved.

---

## Phase 3 Scope

Phase 3 implements controlled EEA historical air quality batch ingestion.
It is limited to the **8 starter cities** and the **3 core pollutants**
(PM2.5, PM10, NO2) defined in Phase 2.

Phase 3 **must not** implement:

- Wikipedia scraping
- Open-Meteo API client behaviour
- Kafka producer logic
- Spark Structured Streaming
- Gold tables
- Dashboards, Airflow, dbt, PostgreSQL, cloud deployment, or ML

Historical EEA batch data is **not** the same as Open-Meteo live/current API
data. These two data contexts must remain separated throughout the pipeline.
Silver and Gold tables must use a `source` field to make this distinction
explicit (`eea` vs `open_meteo`).

---

## Phase 3 EEA Source Access (Issue 3.1)

This section documents how EEA raw data is obtained, stored locally, and
kept out of the repository. It is documentation and hygiene only. No data
download, loader implementation, Spark job, or Silver/Gold output belongs
here.

Full policy details are in `docs/data_sources.md` under the
**Phase 3 EEA Source Access** heading.

---

### EEA Source Access Path

EEA historical air quality time series are available through:

| Access path | URL | Phase 3 use |
| --- | --- | --- |
| EEA Air Quality Download web app | `https://eeadmz1-downloads-webapp.azurewebsites.net` | Select country, station, pollutant, year range; download Parquet or CSV. |
| EEA station spatial service | ArcGIS REST layer at `air.discomap.eea.europa.eu` | Re-query for specific station EoI codes identified in Phase 1. |
| Station-specific Parquet links | Embedded in EEA station popup metadata | Preferred access path for per-station, per-pollutant E1a validated time series. |

Phase 1 already identified candidate stations for Vienna (`AT90TAB`, `AT90AKC`)
and Berlin (`DEBE068`). Phase 3 must extend station review to all 8 starter
cities before ingestion logic is written (Issue 3.3).

---

### Raw-Data Policy

| Rule | Detail |
| --- | --- |
| Large raw EEA files **must not** be committed | `data/**/*.parquet`, `data/**/*.csv`, `data/**/*.json` are git-ignored. |
| Raw files live under `data/bronze/eea/` | This directory is git-ignored. `.gitkeep` maintains the folder structure. |
| Raw files must be reproducibly referenced | Document the station ID, pollutant, year range, and download endpoint so any reviewer can re-download the same files. |
| Tiny test fixtures are allowed | Small in-memory or temporary pytest fixtures only; not real bulk data; not committed. |
| No bulk downloads | Only the stations and time periods needed for 8 cities and 3 pollutants. |

---

### Naming Convention For Local EEA Files

Files stored under `data/bronze/eea/` must follow this pattern:

```
eea_<station_id>_<pollutant_key>_<year_start>_<year_end>.<ext>
```

Examples:

```
eea_AT90TAB_pm25_2018_2023.parquet
eea_AT90TAB_no2_2018_2023.parquet
eea_DEBE068_pm10_2020_2024.csv
```

Where `<pollutant_key>` is one of `pm25`, `pm10`, `no2`.  
Tiny local validation samples may use a `sample_` prefix:

```
sample_eea_AT90TAB_pm25_2022.csv
```

---

### Issue 3.1 Definition Of Done

- [x] EEA source access path is documented.
- [x] Raw-data policy for EEA files is documented.
- [x] Naming convention for local EEA samples is documented.
- [x] `.gitignore` behaviour is confirmed (all `data/**/*.csv`, `*.parquet`, `*.json` are ignored).
- [x] Scope boundary is explicit: no bulk download, no loader, no Spark, no Gold output.
- [x] `docs/data_sources.md` contains `Phase 3 EEA Source Access` section.
- [x] This notebook contains a matching Markdown summary.

---

## EEA Input Schema And Silver Output Schema (Issue 3.2)

This section documents the schema contract for Phase 3 before any
transformation logic is implemented. It defines expected EEA input fields,
pollutant normalisation, the Silver output schema, and the historical vs.
live data separation rule.

Full details are in `docs/data_model.md` under
**Phase 3 EEA Batch Ingestion Data Model**.

---

### EEA Input Field Expectations

EEA station-level files expose the following source concepts that Phase 3
must map to internal canonical fields:

| Source concept | Expected EEA field(s) | Notes |
| --- | --- | --- |
| Station identity | `AirQualityStation`, `AirQualityStationEoICode` | Used for station-to-city mapping; never used as downstream city join key. |
| Measurement timestamp | `DatetimeBegin`, `DatetimeEnd` | Exact name verified on first real file. UTC normalisation required. |
| Pollutant label | `AirPollutant` | Must be normalised to canonical internal name (see table below). |
| Measured value | `Concentration` | Must be non-negative after quality filtering. |
| Unit | `Unit` | Expected `µg/m³`; must be verified per file. |
| Validity / quality flag | `Validity` | Known-bad rows must be excluded before aggregation. |

> Column names must be verified against a real controlled sample file in
> Issue 3.4. If the real file exposes different names, the loader maps them
> without changing this schema contract.

---

### Pollutant Normalisation

Phase 3 processes **exactly 3 core pollutants**. EEA source labels must be
normalised to these internal canonical names before any aggregation:

| Internal canonical name | Accepted EEA source labels | Unit |
| --- | --- | --- |
| `PM2.5` | `PM2.5`, `Particles < 2.5 µm (aerodynamic diameter)`, `PM2,5` | `µg/m³` |
| `PM10` | `PM10`, `Particles < 10 µm (aerodynamic diameter)` | `µg/m³` |
| `NO2` | `NO2`, `Nitrogen dioxide (air)`, `Nitrogen dioxide` | `µg/m³` |

Any record whose pollutant does not match one of these three must be
**silently excluded**. No other pollutants are in scope.

---

### EEA Silver City Daily Schema

**Output path:** `data/silver/eea_city_daily.parquet`

This Parquet file aggregates station-level measurements to
city/day/pollutant granularity using the `city_id` join key from
`data/silver/city_reference.parquet`.

| field | type | required | nullability | rule |
| --- | --- | --- | --- | --- |
| `city_id` | string | yes | non-null | Canonical join key from `city_reference.parquet`. Never a free-text city name. |
| `date` | date | yes | non-null | UTC-normalised daily measurement date. |
| `pollutant` | string | yes | non-null | One of `PM2.5`, `PM10`, `NO2` (canonical internal names only). |
| `mean_value` | float | yes | non-null | Daily mean concentration across all valid observations. |
| `min_value` | float | no | nullable | Daily minimum. Null if only one observation is available. |
| `max_value` | float | no | nullable | Daily maximum. Null if only one observation is available. |
| `observation_count` | integer | yes | non-null | Count of valid, non-negative observations used. Must be ≥ 1. |
| `unit` | string | yes | non-null | Normalised unit; expected `µg/m³`. |
| `source` | string | yes | non-null | Always `eea`. Distinguishes historical EEA batch from live Open-Meteo data. |
| `processing_time_utc` | timestamp | yes | non-null | UTC timestamp when the row was generated. Traceability field. |

#### Key Validation Constraints

- `city_id` must exist in `city_reference.parquet`; unmatched rows are excluded.
- `mean_value` must be ≥ 0; negative values indicate invalid measurements.
- `observation_count` must be ≥ 1; zero-observation rows are not written.
- `source` must be the literal string `eea`.

---

### Historical vs. Live Data Separation

EEA batch data and Open-Meteo live data are **different data contexts**
and must never be silently merged:

| Attribute | Historical EEA | Live Open-Meteo |
| --- | --- | --- |
| Data type | Historical validated measurements | Current / forecast API data |
| Temporal coverage | Multi-year historical records | Current window (days to weeks) |
| Spatial basis | Station-based → mapped to `city_id` | Coordinate-based → mapped to `city_id` |
| Silver table | `data/silver/eea_city_daily.parquet` | `data/silver/open_meteo_city_hourly/` |
| `source` field value | `eea` | `open_meteo` |

No Phase 3 code may read from, write to, or join with Open-Meteo Silver or
Gold tables. The `source` field makes this separation machine-checkable.

---

### Issue 3.2 Definition Of Done

- [x] EEA input field expectations documented.
- [x] Pollutant normalisation table defined (PM2.5, PM10, NO2 only).
- [x] Silver output schema documented with all required fields, types, nullability, and validation rules.
- [x] Historical vs. live data separation documented.
- [x] `docs/data_model.md` contains `Phase 3 EEA Batch Ingestion Data Model` section.
- [x] This notebook contains matching Markdown schema summary.
- [x] No Spark job, Kafka work, Open-Meteo merge, or Gold table was implemented in Issue 3.2.

---

## EEA Station-To-City Mapping Table (Issue 3.3)

This section documents the controlled station-to-city mapping structure
implemented in Phase 3 Issue 3.3. The mapping is built from local constants
in `src/city_mapping/build_station_mapping.py` and does not download EEA
station metadata or trigger any external network calls.

Full details are in `docs/data_model.md` under
**Phase 3.3 EEA Station-To-City Mapping Table**.

---

### Mapping Structure

Each mapping row links one EEA station to one `city_id` from
`city_reference.parquet`. The mapping is the bridge between raw EEA
station-level measurements and city-level Silver Parquet output.

**Output path (local, git-ignored):**
`data/silver/eea_station_city_mapping.parquet`

| field | required | rule |
| --- | --- | --- |
| `city_id` | yes | Canonical join key; must exist in `city_reference.parquet`. |
| `eea_station_id` | yes | Stable EEA station EoI code. Never a free-text name. |
| `station_latitude` | yes* | WGS84 latitude; nullable for placeholder entries. |
| `station_longitude` | yes* | WGS84 longitude; nullable for placeholder entries. |
| `distance_km_to_city_center` | yes* | Haversine distance; computed for known coordinates. |
| `pollutants_available` | yes | PM2.5, PM10, NO2 coverage observed. |
| `mapping_status` | yes | One of `selected`, `candidate`, `fallback`, `rejected`. |
| `representativeness_notes` | yes | Reviewer note on station suitability. |
| `mapping_notes` | yes | Transparent rationale for the mapping decision. |

---

### Current Station Status

| city_id | Station | Status | Notes |
| --- | --- | --- | --- |
| `vienna_at` | `AT90TAB` Taborstrasse | **selected** | PM2.5, PM10, NO2 from 2013-2024. Primary station. |
| `vienna_at` | `AT90AKC` AKH | candidate | All 3 pollutants; fallback if AT90TAB insufficient. |
| `vienna_at` | `AT9STEF` Stephansplatz | candidate | NO2 only; not suitable as primary. |
| `berlin_de` | `DEBE068` Berlin Mitte | **selected** | NO2+PM10 from 2013; PM2.5 from 2020 only. |
| `paris_fr` | PLACEHOLDER | candidate | Station not yet reviewed. |
| `madrid_es` | PLACEHOLDER | candidate | Station not yet reviewed. |
| `rome_it` | PLACEHOLDER | candidate | Station not yet reviewed. |
| `amsterdam_nl` | PLACEHOLDER | candidate | Station not yet reviewed. |
| `warsaw_pl` | PLACEHOLDER | candidate | Station not yet reviewed. |
| `prague_cz` | PLACEHOLDER | candidate | Station not yet reviewed. |

> **Constraint:** Phase 3 ingestion (Issue 3.4) must not proceed for any
> city whose primary station is still a PLACEHOLDER.

---

### Berlin PM2.5 Constraint

`DEBE068` has PM2.5 data only from 2020 onwards. Historical PM2.5
comparisons for Berlin before 2020 are not possible from this station.
This constraint must be documented in any Silver or Gold table that
includes Berlin PM2.5.

---

### Issue 3.3 Definition Of Done

- [x] Station mapping structure implemented as a local Python builder.
- [x] `build_station_mapping()` is deterministic and side-effect free on import.
- [x] All 8 starter cities have at least one mapping entry.
- [x] Pilot city stations (Vienna `AT90TAB`, Berlin `DEBE068`) are `selected`.
- [x] `validate_station_mapping()` checks `city_id` integrity against `city_reference.parquet`.
- [x] `mapping_status` restricted to `selected`, `candidate`, `fallback`, `rejected`.
- [x] Haversine distance computed for stations with known coordinates.
- [x] Unresolved placeholder cities documented with explicit query instructions.
- [x] `docs/data_model.md` contains Phase 3.3 section.
- [x] 14 new station mapping tests added; all 29 tests pass.
- [x] No bulk EEA download, loader, Spark, Kafka, or Gold table implemented.

---

## EEA Batch Loader For Controlled Local Files (Issue 3.4)

This section documents the EEA batch loader implemented in
`src/ingestion/eea_loader.py`. The loader reads local EEA CSV or Parquet
files, normalises source columns to canonical fields, filters to the three
core pollutants, and produces the Silver schema output.

---

### Loader Functions

| Function | Purpose |
| --- | --- |
| `load_eea_raw(path)` | Read a local CSV or Parquet file; resolve column aliases; filter to PM2.5, PM10, NO2; exclude invalid rows. Returns `RAW_COLUMNS` DataFrame. |
| `map_stations_to_cities(raw_df, station_mapping_df)` | Join raw records to `city_id` via `selected` station mapping entries only. Drops rows with no selected mapping. |
| `aggregate_to_city_daily(mapped_df)` | Group by `city_id`, `date`, `pollutant`, `unit`; compute mean/min/max/count; add `source='eea'` and `processing_time_utc`. Returns Silver schema DataFrame. |
| `load_and_aggregate(path, station_mapping_df)` | Convenience: combines all three steps into one call. |

---

### Column Alias Resolution

EEA files may expose different column names depending on export format.
The loader resolves these aliases in order and raises `KeyError` if none match:

| Concept | Accepted aliases |
| --- | --- |
| Station identity | `AirQualityStationEoICode`, `AirQualityStation`, `station_id` |
| Measurement timestamp | `DatetimeBegin`, `datetime_begin`, `Start`, `date` |
| Pollutant label | `AirPollutant`, `pollutant`, `Pollutant`, `Component` |
| Measured value | `Concentration`, `concentration`, `Value`, `value` |
| Unit | `Unit`, `unit` |
| Validity flag | `Validity`, `validity`, `DataValid` (optional) |

---

### Quality Filters Applied By `load_eea_raw`

| Filter | Rule |
| --- | --- |
| Pollutant scope | Only PM2.5, PM10, NO2 (canonical internal names); all others excluded. |
| Pollutant label normalisation | Source labels mapped via `POLLUTANT_LABEL_MAP` before filtering. |
| Timestamp validity | Rows with unparseable timestamps excluded; remaining converted to UTC. |
| Concentration validity | Rows with negative or missing concentration excluded. |
| Validity flag | Rows with `Validity` in `{-1, -99, -999}` excluded if column present. |

---

### Station Mapping Constraint

`map_stations_to_cities` only uses mapping entries with
`mapping_status = 'selected'`. Records for `candidate`, `fallback`, or
`rejected` stations are excluded. This prevents unreviewed placeholder
stations from entering the Silver layer.

---

### Issue 3.4 Definition Of Done

- [x] `src/ingestion/eea_loader.py` imports without side effects.
- [x] Loader reads local CSV and Parquet from explicit caller-provided paths.
- [x] Column aliases resolved in order; `KeyError` raised with clear message if none match.
- [x] Pollutant label normalisation via `POLLUTANT_LABEL_MAP` (verbose EEA labels supported).
- [x] Filters to PM2.5, PM10, NO2 only; all other pollutants silently excluded.
- [x] Excludes negative and missing concentration values.
- [x] Excludes rows with known-bad validity flags (`-1`, `-99`, `-999`).
- [x] Excludes rows with unparseable timestamps; converts remaining to UTC.
- [x] `map_stations_to_cities` uses only `selected` station entries.
- [x] `aggregate_to_city_daily` returns Silver schema with `source='eea'`.
- [x] No `requests`, `urllib`, `kafka`, `pyspark`, or `SparkSession` imports.
- [x] 39 loader tests pass; 71 total tests pass.
- [x] No raw EEA data committed; all tests use tiny in-memory/tmp_path fixtures.

---

## EEA Data Quality Validation Rules (Issue 3.5)

Phase 3 validates normalized EEA measurement rows before daily aggregation. The goal is to prevent invalid station records from silently entering `data/silver/eea_city_daily.parquet`.

### Required row-level fields

| field | rule |
| --- | --- |
| `city_id` | Required, non-null canonical city join key from station mapping. |
| `datetime_begin` | Required, parseable timestamp normalized to UTC. |
| `pollutant` | Required, one of PM2.5, PM10, NO2. |
| `concentration` | Required numeric measurement, non-negative after validation. |
| `unit` | Required non-empty unit string, expected micrograms per cubic metre or equivalent source spelling. |

Missing required columns fail validation with `ValueError`. Null or empty required fields also fail validation.

### Invalid-row handling

| data quality issue | behavior |
| --- | --- |
| Negative concentration | Rejected before aggregation. |
| Missing or non-numeric concentration | Rejected before aggregation. |
| Unsupported pollutant outside PM2.5, PM10, NO2 | Rejected before aggregation. |
| Invalid or unparseable timestamp | Validation fails; source file requires review. |
| Missing unit | Validation fails; source file requires review. |
| Missing `city_id` | Validation fails; station mapping must be fixed. |
| Known invalid validity flag (`-1`, `-99`, `-999`) | Rejected during raw loading. |

### Implementation notes

`src/ingestion/eea_loader.py` exposes `validate_eea_rows(df)`. The aggregation path calls this validator before grouping by `city_id`, `date`, `pollutant`, and `unit`.

These rules do not implement Gold analytics, live API validation, Spark streaming, or causal interpretation.

### Issue 3.5 Definition Of Done

- [x] Data quality function exists and is tested.
- [x] Negative or invalid measurement values are rejected before aggregation.
- [x] Unsupported pollutants do not pass into Phase 3 output.
- [x] Missing required fields fail validation.
- [x] Data quality limitations are documented.


## EEA City Daily Silver Parquet Output (Issue 3.6)

Issue 3.6 implements the local Parquet write boundary for historical EEA Silver output.

**Output path:** `data/silver/eea_city_daily.parquet`

The output is generated only when `write_eea_city_daily_parquet()` or `build_eea_city_daily_parquet()` is explicitly called. Importing `src/ingestion/eea_loader.py` does not write files.

### Aggregation grain

Rows are grouped by:

- `city_id`
- UTC `date`
- `pollutant`
- `unit`

For each group the loader calculates `mean_value`, `min_value`, `max_value`, and `observation_count`, then adds `source = "eea"` and `processing_time_utc`.

### Silver output boundary

The Parquet writer validates that output columns match the documented Silver schema, pollutants are limited to PM2.5, PM10, and NO2, and the source field contains only `eea`.

The generated Parquet file is ignored by Git via `data/**/*.parquet`. It is a local reproducible Phase 3 deliverable, not committed source data.

### Issue 3.6 Definition Of Done

- [x] Aggregation is deterministic for controlled input rows.
- [x] Output schema matches `docs/data_model.md`.
- [x] Output contains only core pollutants.
- [x] Parquet is written only when explicitly called.
- [x] Parquet readback is covered by tests.
- [x] Historical EEA output remains separated from Open-Meteo live/API data.

---

## Reading The Silver Output Once Generated (Issue 3.7)

The Phase 3 Silver output is a local, git-ignored Parquet file. It is generated only when a controlled local EEA input file and selected station mapping are passed to the explicit writer functions.

Expected output path:

```text
data/silver/eea_city_daily.parquet
```

Tiny local readback example after generation:

```python
from pathlib import Path
import pandas as pd

p = Path("data/silver/eea_city_daily.parquet")
if p.exists():
    eea_daily = pd.read_parquet(p)
    expected = {
        "city_id", "date", "pollutant", "mean_value", "min_value",
        "max_value", "observation_count", "unit", "source",
        "processing_time_utc",
    }
    assert expected.issubset(eea_daily.columns)
```

Do not display large raw EEA files or large Silver tables in this notebook. Use `head()` only for tiny local inspection if a reviewer explicitly needs a visual check.

### Issue 3.7 Definition Of Done

- [x] Phase 3 scope and boundaries documented.
- [x] EEA source policy and raw-data hygiene documented.
- [x] EEA schema and core pollutants documented.
- [x] Station-to-city mapping dependency documented.
- [x] Data quality rules documented.
- [x] Daily aggregation and output contract documented.
- [x] Readback pattern for `eea_city_daily.parquet` documented without executing large outputs.
- [x] Historical EEA data is explicitly separated from Open-Meteo live/API data.

---

## Phase 3 Pending Work (Issue 3.8)

The remaining Phase 3 work is QA only:

| Issue | Title | Status |
| --- | --- | --- |
| 3.3 | Prepare EEA station-to-city mapping table | **done** |
| 3.4 | Implement EEA loader for controlled local files | **done** |
| 3.5 | Add EEA data quality validation rules | **done** |
| 3.6 | Build EEA city daily Silver Parquet | **done** |
| 3.7 | Update this notebook with full Phase 3 documentation | **done** |
| 3.8 | Phase 3 QA report and gate decision | pending |

---

## Phase 3 Deliverables

| Artefact | Path | Status |
| --- | --- | --- |
| This notebook | `notebooks/02_eea_batch_ingestion.ipynb` | **complete for Phase 3 implementation trail** (Issue 3.7) |
| Source access documentation | `docs/data_sources.md` | Issue 3.1 section added |
| EEA + Silver schema documentation | `docs/data_model.md` | Issue 3.2 section added |
| EEA Loader | `src/ingestion/eea_loader.py` | **done** (Issue 3.4) |
| Batch Processing Job | `src/spark_jobs/batch_eea_processing_job.py` | placeholder only; Phase 3 uses pandas/pyarrow local writer |
| Silver Output | `data/silver/eea_city_daily.parquet` | **implemented as explicit local writer** (Issue 3.6) |
| Data Quality Summary | `docs/data_sources.md` and this notebook | **done** (Issue 3.5) |
| Phase 3 QA Report | `docs/qa/phase3_qa_report.md` | pending (Issue 3.8) |